In [2]:
import pandas as pd
import os
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from scipy.sparse import load_npz

In [2]:
X_train=load_npz("X_train.npz")
X_test=load_npz("X_test.npz")

y_train=np.load("Y_train.npy")
y_test=np.load("Y_test.npy")

In [3]:
X_small = X_train[:20000]
y_small = y_train[:20000]

X_small_test = X_test[:20000]
y_small_test = y_test[:20000]

In [4]:
param_grid = {
    'C': [0.1, 1, 3, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

scoring = {
    'auc': 'roc_auc',
    'accuracy': 'accuracy',
    'f1': 'f1'
}

model = GridSearchCV(
    estimator=LinearSVC(max_iter=10000),  
    param_grid=param_grid,
    cv=cv,
    scoring=scoring,
    refit='f1',              
    n_jobs=-1,
    verbose=3
)

model.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,LinearSVC(max_iter=10000)
,param_grid,"{'C': [0.1, 1, ...]}"
,scoring,"{'accuracy': 'accuracy', 'auc': 'roc_auc', 'f1': 'f1'}"
,n_jobs,-1
,refit,'f1'
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [6]:
pd.set_option('display.max_columns', None)
pd.DataFrame(model.cv_results_).sort_values('rank_test_f1')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,params,split0_test_auc,split1_test_auc,split2_test_auc,split3_test_auc,split4_test_auc,mean_test_auc,std_test_auc,rank_test_auc,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,split4_test_accuracy,mean_test_accuracy,std_test_accuracy,rank_test_accuracy,split0_test_f1,split1_test_f1,split2_test_f1,split3_test_f1,split4_test_f1,mean_test_f1,std_test_f1,rank_test_f1
3,84.317880,3.298262,0.232130,0.171233,5.0,{'C': 5},0.999201,0.999298,0.999535,0.999263,0.999619,0.999383,0.000164,1,0.991679,0.992316,0.993798,0.992385,0.994625,0.992961,0.001083,1,0.991665,0.992302,0.993786,0.992373,0.994615,0.992948,0.001084,1
2,82.566897,3.767968,0.650624,0.466027,3.0,{'C': 3},0.999203,0.999600,0.999297,0.999587,0.998520,0.999241,0.000393,2,0.991610,0.993781,0.991765,0.994177,0.987630,0.991793,0.002324,2,0.991596,0.993773,0.991752,0.994169,0.987590,0.991776,0.002336,2
1,83.693861,2.405786,0.473853,0.264322,1.0,{'C': 1},0.998932,0.999399,0.999614,0.998580,0.999211,0.999147,0.000361,3,0.989301,0.993040,0.994280,0.988164,0.992402,0.991438,0.002318,3,0.989273,0.993027,0.994269,0.988137,0.992387,0.991419,0.002325,3
0,80.710654,6.824625,0.454753,0.340179,0.1,{'C': 0.1},0.998578,0.999167,0.998693,0.999170,0.998550,0.998832,0.000279,4,0.987062,0.991955,0.987441,0.992282,0.987665,0.989281,0.002327,4,0.987026,0.991933,0.987403,0.992269,0.987627,0.989252,0.002337,4


In [14]:
best_model = model.best_estimator_

y_pred = best_model.predict(X_small_test)

test_accuracy = accuracy_score(y_small_test, y_pred)
test_f1 = f1_score(y_small_test, y_pred)
test_auc = roc_auc_score(y_small_test, best_model.decision_function(X_small_test))

print("Test Accuracy:", test_accuracy)
print("Test F1 Score:", test_f1)
print("Test AUC:", test_auc)

Test Accuracy: 0.99425
Test F1 Score: 0.9942225571464456
Test AUC: 0.999692418228329
